# Dependencias

In [27]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import time

from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Nos fijamos si vamos a usar GPU o CPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


In [28]:
# Fijamos la semilla 
torch.manual_seed(100)
if torch.cuda.is_available():
    torch.cuda.manual_seed(100)

In [29]:
def normas_de_gradiente(modelo):
    """Norma del gradiente de los pesos de cada capa (convolusiones y densas)."""
    normas = {}
    for nombre, parametro in modelo.named_parameters():
        es_peso = nombre.endswith("weight")
        es_capa_de_interes = ("conv" in nombre) or ("fc" in nombre)   # se ignora la batch normalization
        if parametro.grad is not None and es_peso and es_capa_de_interes:
            normas[nombre] = parametro.grad.detach().norm(2).item()
    return normas

# Importando dataset

In [30]:
dataset = torchvision.datasets.EuroSAT(
    root="./data",
    download=True,
)

clases = dataset.classes
num_clases = len(clases)

print("Clases:", clases)
print("Cantidad de clases:", num_clases)
print("Cantidad de imágenes:", len(dataset))

Clases: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Cantidad de clases: 10
Cantidad de imágenes: 27000


In [31]:
indices = np.arange(len(dataset))
etiquetas = dataset.targets

idx_train, idx_temp = train_test_split(
    indices,
    test_size=0.30,
    stratify=etiquetas,
    random_state=100
)

idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=0.50,
    stratify=np.array(etiquetas)[idx_temp],
    random_state=100
)

# Ahora si -- esto es lo que va en `desarrollo.ipynb`

Nuestra elección de modelo para realizar transfer learning es ResNet18.

Dado que ResNet18 fue entrenado con ImageNet, cuya media y error son `[0.485, 0.456, 0.406]` y `[0.229, 0.224, 0.225]` respectivamente, vamos a tener que usar estos valores para la normalización.

Además, vamos a tener que redimensionar las imagenes, ya que este modelo trabaja con tamaños de $224 \times 224$, mientras que las imagenes de EuroSAT RGB son de $64 \times 64$.

Con esto preparamos los datos.

In [32]:
# Normalización de ImageNet
mean_imagenet = [0.485, 0.456, 0.406]
std_imagenet = [0.229, 0.224, 0.225]

# Para entrenamiento
transform_train_resnet = transforms.Compose([
    transforms.Resize((224, 224)), # redimensionamos imagenes de 64x64 a 224x224
    transforms.RandomHorizontalFlip(), # el resto es igual a lo que usamos para nuestro modelo
    transforms.RandomVerticalFlip(),
    transforms.RandomChoice([
        transforms.RandomRotation((0, 0)),
        transforms.RandomRotation((90, 90)),
        transforms.RandomRotation((180, 180)),
        transforms.RandomRotation((270, 270))
    ]),
    transforms.ToTensor(),
    transforms.Normalize( # normalización de valores, ahora con los valores de ImageNet
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Para validación y test
transform_eval_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

Aplicamos las transformaciones.

In [33]:
dataset_train_resnet = torchvision.datasets.EuroSAT(
    root="./data",
    transform=transform_train_resnet
)

dataset_eval_resnet = torchvision.datasets.EuroSAT(
    root="./data",
    transform=transform_eval_resnet
)

# aca reutilizamos los indices idx_(...) para garantizar que ambos modelos usen las mismas imagenes
train_data_resnet = Subset(dataset_train_resnet, idx_train)
val_data_resnet = Subset(dataset_eval_resnet, idx_val)
test_data_resnet = Subset(dataset_eval_resnet, idx_test)

Creamos los DataLoaders.

In [34]:
batch_size = 64

train_loader_resnet = DataLoader(
    train_data_resnet, 
    batch_size=batch_size, 
    shuffle=True,                      
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader_resnet = DataLoader(
    val_data_resnet, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=2, 
    pin_memory=True, 
    persistent_workers=True
)

test_loader_resnet = DataLoader(
    test_data_resnet, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=2, 
    pin_memory=True, 
    persistent_workers=True
)

Cargando y adaptando ResNet.

In [35]:
from torchvision.models import resnet18, ResNet18_Weights

# cargamos ResNet18 con pesos preentrenados en ImageNet
weights = ResNet18_Weights.DEFAULT
modelo_resnet = resnet18(weights=weights)

# congelamos todos los parámetros
for param in modelo_resnet.parameters():
    param.requires_grad = False

# descongelamos el último bloque residual
for param in modelo_resnet.layer4.parameters():
    param.requires_grad = True

# reemplazamos la capa clasificadora, que aprende a distinguir las 10 clases de EuroSat
modelo_resnet.fc = nn.Linear(
    modelo_resnet.fc.in_features,
    num_clases
)

# enviamos el modelo al dispositivo
modelo_resnet = modelo_resnet.to(device)

Y ahora lo entrenamos con EuroSat:

In [36]:
learning_rate = 0.01
momentum = 0.9

criterio_opt = nn.CrossEntropyLoss()
# como ResNet viene preentrenada layer4 usa un lr más chico
optimizador = optim.SGD([
    {"params": modelo_resnet.layer4.parameters(), "lr": learning_rate / 10},
    {"params": modelo_resnet.fc.parameters(),     "lr": learning_rate},
], momentum = momentum)

In [39]:
cant_epocas = 4 # valor chico para testear,..

# historial para graficar y comparar
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
tiempos_por_epoca = []
normas_grad_por_epoca = []   # solo las primeras épocas

for epoca in range(cant_epocas):

    t0 = time.time()
    modelo_resnet.train()

    running_loss = 0.0
    train_correct = 0
    total = 0
    normas_epoca = []
    for i, data in enumerate(train_loader_resnet, 0):

        optimizador.zero_grad() # Reiniciamos los gradientes.

        outputs = modelo_resnet(data[0].to(device)) # Hacemos la pasada forward.
        loss = criterio_opt(outputs, data[1].to(device)) # Calculamos la pérdida.
        loss.backward() # Hacemos la pasada backward.

        # ---- esto es para comparar despues, habria que implementarlo en nuestro modelo 

        if epoca < 2: # Monitoreo de gradientes: después del backward y antes del step.
            normas_epoca.append(normas_de_gradiente(modelo_resnet))

        # --------------------------------------------------------------

        optimizador.step() # Actualizamos los pesos.

        running_loss += loss.item() * (data[0].to(device)).shape[0]

        _, predicted = torch.max(outputs.data, 1)
        total += (data[1].to(device)).size(0)
        train_correct += (predicted == (data[1].to(device))).sum().item()

    train_accuracy = 100 * train_correct / total
    running_loss = running_loss / total

    # Veamos en validación.
    modelo_resnet.eval()
    val_correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad(): # Esta parte es sólo validación, no requiere entrenar, por lo que no calculamos gradientes.
        for data in val_loader_resnet:
            outputs = modelo_resnet(data[0].to(device))
            _, predicted = torch.max(outputs.data, 1) # La ganadora es la que tiene mayor valor.
            total += (data[1].to(device)).size(0)
            val_correct += (predicted == (data[1].to(device))).sum().item()
            val_loss += criterio_opt(outputs, data[1].to(device)).item() * data[0].to(device).shape[0]

    val_accuracy = 100 * val_correct / total
    val_loss = val_loss / total


    # ---- esto es para comparar despues, habria que implementarlo en nuestro modelo 

    if device.type == "cuda":
        torch.cuda.synchronize()
    tiempo_epoca = time.time() - t0

    # Guardamos el historial
    train_losses.append(running_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)
    tiempos_por_epoca.append(tiempo_epoca)
    if normas_epoca:
        normas_grad_por_epoca.append(normas_epoca)

    # --------------------------------------------------------------


    print(f'Época {epoca + 1 } terminada. Train accuracy: {train_accuracy:.2f}%. Validation accuracy: {val_accuracy:.2f}%. Tiempo: {tiempo_epoca:.1f}s.')

print('Entrenamiento terminado.')

RuntimeError: DataLoader worker (pid(s) 66804, 66813) exited unexpectedly